## Imports


In [11]:
import gymnasium as gym
import math
import random
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
from collections import namedtuple, deque
from itertools import count

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import ale_py
from gymnasium.wrappers import AtariPreprocessing, FrameStackObservation
gym.register_envs(ale_py)


## Replay Memory for NN training

In [12]:
env = gym.make("ALE/Phoenix-v5", frameskip=1, full_action_space=True)
env = AtariPreprocessing(env, grayscale_obs=True, scale_obs=False, frame_skip=4)
env = FrameStackObservation(env, stack_size=4)

# set up matplotlib
is_ipython = 'inline' in matplotlib.get_backend()
if is_ipython:
    from IPython import display

plt.ion()

# if GPU is to be used
device = torch.device(
    "cuda" if torch.cuda.is_available() else
    "mps" if torch.backends.mps.is_available() else
    "cpu"
)

Transition = namedtuple('Transition',('state', 'action', 'next_state', 'reward'))

In [13]:
class ReplayMemory(object):

    def __init__(self, capacity):
        self.memory = deque([], maxlen=capacity)

    def push(self, *args):
        """Save a transition"""
        self.memory.append(Transition(*args))

    def sample(self, batch_size):
        return random.sample(self.memory, batch_size)

    def __len__(self):
        return len(self.memory)

## DQN Network ( 3- layerd feefforward Network )

In [14]:
class DQN(nn.Module):
    def __init__(self, input_shape, n_actions):
        super().__init__()
        c, h, w = input_shape  # e.g. (4, 84, 84)

        self.conv = nn.Sequential(
            nn.Conv2d(c, 32, kernel_size=8, stride=4), nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2), nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1), nn.ReLU(),
        )

        with torch.no_grad():
            dummy = torch.zeros(1, c, h, w)
            conv_out = self.conv(dummy).view(1, -1).size(1)

        self.head = nn.Sequential(
            nn.Linear(conv_out, 512), nn.ReLU(),
            nn.Linear(512, n_actions)
        )

    def forward(self, x):
        # x: (batch, C, H, W), uint8 or float
        x = x.float() / 255.0
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return self.head(x)


## Network Training

In [15]:
BATCH_SIZE = 128
GAMMA = 0.99
EPS_START = 0.9
EPS_END = 0.01
EPS_DECAY = 2500
TAU = 0.005
LR = 3e-4


# Get number of actions from gym action space
n_actions = env.action_space.n
# Get the number of state observations
state, info = env.reset()
n_observations = env.observation_space.shape # e.g. (4, 84, 84) after wrappers

policy_net = DQN(n_observations, n_actions).to(device)
target_net = DQN(n_observations, n_actions).to(device) # This is used to compute the Target TD, the weights for this are usually frozen
target_net.load_state_dict(policy_net.state_dict()) # copies all the weights and biases of the police_net to the target_net

optimizer = optim.AdamW(policy_net.parameters(), lr=LR, amsgrad=True)
memory = ReplayMemory(10000)

steps_done = 0
episode_durations = []


In [16]:
def select_action(state):
    """
    we pick an epislon greedy action
    :param state:
    :return:
    a tensor (1,1) of d_type long, this is needed to ensure torch.gather works later while selecting the Q values from (batch_size, N-actions) from the output of policy_net(states)
    We need thus for the Bellman Loss:
    """
    global steps_done
    sample = random.random()
    # construct the eps decay based on the number of steps done
    eps_threshold = EPS_END + (EPS_START - EPS_END) * math.exp(-1. * steps_done / EPS_DECAY)
    steps_done += 1
    if sample > eps_threshold:
        with torch.no_grad():
            # t.max(1) will return the largest column value of each row.
            # second column on max result is index of where max element was
            # found, so we pick action with the larger expected reward.
            return policy_net(state).max(1).indices.view(1, 1) # returns a named tuple with .values and .indices
    else:
        return torch.tensor([[env.action_space.sample()]], device=device, dtype=torch.long)

In [17]:
def plot_durations(show_result=False):
    global episode_durations
    plt.figure(1)
    durations_t = torch.tensor(episode_durations, dtype=torch.float)
    if show_result:
        plt.title('Result')
    else:
        plt.clf()
        plt.title('Training...')
    plt.xlabel('Episode')
    plt.ylabel('Duration')
    plt.plot(durations_t.numpy())
    # Take 20 episode averages and plot them too
    if len(durations_t) >= 20:
        means = durations_t.unfold(0, 20, 1).mean(1).view(-1)
        means = torch.cat((torch.zeros(19), means))
        plt.plot(means.numpy())

    plt.pause(0.001)  # pause a bit so that plots are updated
    if is_ipython:
        if not show_result:
            display.display(plt.gcf())
            display.clear_output(wait=True)
        else:
            display.display(plt.gcf())

In [18]:
def optimize_model():
    if len(memory) < BATCH_SIZE:
        return
    transitions = memory.sample(BATCH_SIZE)
    # Transpose the batch (see https://stackoverflow.com/a/19343/3343043 for
    # detailed explanation). This converts batch-array of Transitions
    # to Transition of batch-arrays.
    batch = Transition(*zip(*transitions))

    # Compute a mask of non-final states and concatenate the batch elements
    # (a final state would've been the one after which simulation ended)
    non_final_mask = torch.tensor(tuple(map(lambda s: s is not None,
                                          batch.next_state)), device=device, dtype=torch.bool)
    non_final_next_states = torch.cat([s for s in batch.next_state
                                                if s is not None])
    state_batch = torch.cat(batch.state)
    action_batch = torch.cat(batch.action)
    reward_batch = torch.cat(batch.reward)

    # Compute Q(s_t, a) - the model computes Q(s_t), then we select the
    # columns of actions taken. These are the actions which would've been taken
    # for each batch state according to policy_net
    state_action_values = policy_net(state_batch).gather(1, action_batch)

    # Compute V(s_{t+1}) for all next states.
    # Expected values of actions for non_final_next_states are computed based
    # on the "older" target_net; selecting their best reward with max(1).values
    # This is merged based on the mask, such that we'll have either the expected
    # state value or 0 in case the state was final.
    next_state_values = torch.zeros(BATCH_SIZE, device=device)
    with torch.no_grad():
        next_state_values[non_final_mask] = target_net(non_final_next_states).max(1).values
    # Compute the expected Q values
    expected_state_action_values = (next_state_values * GAMMA) + reward_batch

    # Compute Huber loss
    criterion = nn.SmoothL1Loss()
    loss = criterion(state_action_values, expected_state_action_values.unsqueeze(1))

    # Optimize the model
    optimizer.zero_grad()
    loss.backward()
    # In-place gradient clipping
    torch.nn.utils.clip_grad_value_(policy_net.parameters(), 100)
    optimizer.step()

## Post Training Analysis: Visualisation and Rank-Estimation

In [19]:
def estimate_q_matrix_rank(
    net: DQN,
    n_samples: int = 512,
    tol: float = 1e-6,
):
    states = []

    if len(memory) > 0:
        for tr in memory.memory:
            if tr.state is not None:
                states.append(tr.state.squeeze(0).cpu().numpy())
                if len(states) >= n_samples:
                    break

    if len(states) < n_samples:
        env_tmp = gym.make("Acrobot-v1")
        try:
            while len(states) < n_samples:
                s, _ = env_tmp.reset()
                done = False
                while not done and len(states) < n_samples:
                    states.append(np.array(s, copy=True))
                    a = env_tmp.action_space.sample()
                    s, _, terminated, truncated, _ = env_tmp.step(a)
                    done = terminated or truncated
        finally:
            env_tmp.close()

    states_arr = np.stack(states, axis=0)

    with torch.no_grad():
        s_t = torch.tensor(states_arr, dtype=torch.float32, device=device)
        q_vals = net(s_t).cpu().numpy()

    q_matrix = q_vals
    rank = np.linalg.matrix_rank(q_matrix, tol=tol)

    print(f"Approximate Q-matrix shape: {q_matrix.shape}")
    print(f"Numerical rank (tol={tol}): {rank}")
    print(f"Rank / min(n_states, n_actions): {rank / min(q_matrix.shape):.3f}")

    return q_matrix, rank

In [20]:
def visualise_policy(net, episodes: int = 3):
    env_vis = gym.make("ALE/Phoenix-v5", render_mode="human", frameskip=1, full_action_space=True)
    try:
        for ep in range(episodes):
            state, _ = env_vis.reset()
            state_t = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
            done = False
            steps = 0
            total_reward = 0.0

            while not done:
                with torch.no_grad():
                    q_vals = net(state_t)
                    action = int(q_vals.argmax(dim=1).item())
                obs, reward, terminated, truncated, _ = env_vis.step(action)
                total_reward += float(reward)
                done = terminated or truncated

                if not done:
                    state_t = torch.as_tensor(np.array(obs), device=device).unsqueeze(0)

                steps += 1

            print(f"Episode {ep + 1}: {steps} steps, reward {total_reward:.1f}")
    finally:
        env_vis.close()

## Main Code for trainning

In [ ]:
if torch.cuda.is_available() or torch.backends.mps.is_available():
    num_episodes = 500
else:
    num_episodes = 50

for i_episode in range(num_episodes):
    # Initialize the environment and get its state
    state, info = env.reset()
    state = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
    for t in count():
        action = select_action(state)
        observation, reward, terminated, truncated, _ = env.step(action.item())
        reward = torch.tensor([reward], device=device, dtype=torch.float32)
        done = terminated or truncated

        if terminated:
            next_state = None
        else:
            next_state = torch.as_tensor(np.array(observation), device=device).unsqueeze(0)

        # Store the transition in memory
        memory.push(state, action, next_state, reward)

        # Move to the next state
        state = next_state

        # Perform one step of the optimization (on the policy network)
        optimize_model()

        # Soft update of the target network's weights
        # θ′ ← τ θ + (1 −τ )θ′
        target_net_state_dict = target_net.state_dict()
        policy_net_state_dict = policy_net.state_dict()
        for key in policy_net_state_dict:
            target_net_state_dict[key] = policy_net_state_dict[key] * TAU + target_net_state_dict[key] * (1 - TAU)
        target_net.load_state_dict(target_net_state_dict)

        if done:
            episode_durations.append(t + 1)
            plot_durations()
            break

print('Complete')
plot_durations(show_result=True)

<Figure size 640x480 with 0 Axes>

In [16]:
print("\nVisualising greedy policy with learned DQN...")
visualise_policy(policy_net, episodes=3)


Visualising greedy policy with learned DQN...
Episode 1: 109 steps, reward -108.0
Episode 2: 73 steps, reward -72.0
Episode 3: 96 steps, reward -95.0


In [17]:
print("\nEstimating low-rank structure of Q from DQN...")
estimate_q_matrix_rank(policy_net, n_samples=10000, tol=1e-1)

plt.ioff()
plt.show()


Estimating low-rank structure of Q from DQN...
Approximate Q-matrix shape: (10000, 3)
Numerical rank (tol=0.1): 3
Rank / min(n_states, n_actions): 1.000
